# Подготовка таблицы по клиентам

В этом ноутбуке мы переходим от первичного анализа к подготовке рабочей таблицы, где одна строка соответствует одному клиенту. Эта таблица станет основой для:

- прогноза ухода клиентов;
- сегментации клиентов;
- персональных рекомендаций;
- бизнес-дашбордов.

## Что делаем

1. Загружаем данные и приводим типы.
2. Очищаем ключевые поля.
3. Собираем признаки по заказам.
4. Собираем признаки по логистике и возвратам.
5. Собираем признаки по событиям.
6. Объединяем всё в единую таблицу клиентов.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 200)

BASE_DIR = Path("..")
DATA_PATH = BASE_DIR / "data.csv"
EVENTS_PATH = BASE_DIR / "events.csv"


In [ ]:
def parse_dt(series: pd.Series) -> pd.Series:
    return pd.to_datetime(series, utc=True, errors="coerce", format="mixed")


def q95_clip(series: pd.Series) -> pd.Series:
    upper = series.quantile(0.95)
    return series.clip(upper=upper)


In [ ]:
data_cols = [
    "order_id", "order_item_id", "user_id", "status", "gender", "created_at", "returned_at",
    "shipped_at", "delivered_at", "sale_price", "age", "state", "city", "traffic_source",
    "category", "department", "brand", "product_id", "warehouse_name", "is_loyal"
]
event_cols = [
    "id", "user_id", "session_id", "sequence_number", "created_at", "traffic_source",
    "browser", "uri", "event_type"
]

data = pd.read_csv(DATA_PATH, usecols=data_cols)
events = pd.read_csv(EVENTS_PATH, usecols=event_cols)

for col in ["created_at", "returned_at", "shipped_at", "delivered_at"]:
    data[col] = parse_dt(data[col])
events["created_at"] = parse_dt(events["created_at"])

data.shape, events.shape


## 1. Базовая очистка

Здесь мы не удаляем агрессивно строки, а аккуратно готовим данные так, чтобы не потерять полезный бизнес-сигнал.

In [ ]:
data = data.drop_duplicates(subset=["order_item_id"]).copy()
events = events.drop_duplicates(subset=["id"]).copy()

data["sale_price"] = data["sale_price"].clip(lower=0)
data["city"] = data["city"].fillna("unknown")
data["brand"] = data["brand"].fillna("unknown")
events["user_id"] = events["user_id"].astype("Int64")

data["ship_delay_days"] = (data["shipped_at"] - data["created_at"]).dt.total_seconds() / 86400
data["delivery_days"] = (data["delivered_at"] - data["shipped_at"]).dt.total_seconds() / 86400
data["return_days"] = (data["returned_at"] - data["delivered_at"]).dt.total_seconds() / 86400

data[["ship_delay_days", "delivery_days", "return_days"]] = data[["ship_delay_days", "delivery_days", "return_days"]].apply(q95_clip)
data.head()


## 2. Признаки по заказам

Сначала собираем информацию по заказам, а потом уже агрегируем её до уровня клиента.

In [ ]:
orders = (
    data.groupby("order_id", as_index=False)
    .agg(
        user_id=("user_id", "first"),
        created_at=("created_at", "min"),
        status=("status", "first"),
        order_value=("sale_price", "sum"),
        items_count=("order_item_id", "nunique"),
        main_traffic_source=("traffic_source", "first"),
        is_loyal=("is_loyal", "max"),
    )
)
orders.head()


In [ ]:
customer_orders = (
    orders.groupby("user_id", as_index=False)
    .agg(
        orders_count=("order_id", "nunique"),
        total_revenue=("order_value", "sum"),
        avg_order_value=("order_value", "mean"),
        median_order_value=("order_value", "median"),
        avg_items_per_order=("items_count", "mean"),
        first_order_at=("created_at", "min"),
        last_order_at=("created_at", "max"),
        loyal_flag=("is_loyal", "max"),
    )
)
customer_orders["customer_lifetime_days"] = (
    customer_orders["last_order_at"] - customer_orders["first_order_at"]
).dt.days
customer_orders.head()


## 3. Признаки по доставке и возвратам

In [ ]:
data["is_returned"] = data["returned_at"].notna().astype(int)
data["is_delivered"] = data["delivered_at"].notna().astype(int)
data["is_cancelled"] = data["status"].eq("Cancelled").astype(int)

customer_logistics = (
    data.groupby("user_id", as_index=False)
    .agg(
        returned_items_count=("is_returned", "sum"),
        delivered_items_count=("is_delivered", "sum"),
        cancelled_items_count=("is_cancelled", "sum"),
        return_rate_items=("is_returned", "mean"),
        avg_ship_delay_days=("ship_delay_days", "mean"),
        median_delivery_days=("delivery_days", "median"),
        avg_return_days=("return_days", "mean"),
    )
)
customer_logistics.head()


## 4. Предпочтения по товарам

In [ ]:
favorite_category = (
    data.groupby(["user_id", "category"], as_index=False)
    .agg(items_count=("order_item_id", "count"))
    .sort_values(["user_id", "items_count"], ascending=[True, False])
    .drop_duplicates(subset=["user_id"])
    .rename(columns={"category": "favorite_category", "items_count": "favorite_category_items"})
)
favorite_category.head()


## 5. Признаки по событиям

Здесь важно помнить: часть событий не привязана к `user_id`, поэтому поведенческие признаки будем строить только по тем пользователям, где связь есть.

In [ ]:
events["is_purchase_event"] = events["event_type"].eq("purchase").astype(int)
events["is_product_event"] = events["event_type"].eq("product").astype(int)
events["is_cart_event"] = events["event_type"].eq("cart").astype(int)
events["is_department_event"] = events["event_type"].eq("department").astype(int)

events_linked = events.dropna(subset=["user_id"]).copy()
events_linked.head()


In [ ]:
customer_events = (
    events_linked.groupby("user_id", as_index=False)
    .agg(
        sessions_count=("session_id", "nunique"),
        events_count=("id", "count"),
        product_views=("is_product_event", "sum"),
        cart_events=("is_cart_event", "sum"),
        department_events=("is_department_event", "sum"),
        purchase_events=("is_purchase_event", "sum"),
        first_event_at=("created_at", "min"),
        last_event_at=("created_at", "max"),
    )
)
customer_events["events_per_session"] = customer_events["events_count"] / customer_events["sessions_count"]
customer_events.head()


## 6. Финальная таблица клиентов

In [ ]:
customer_table = (
    customer_orders
    .merge(customer_logistics, on="user_id", how="left")
    .merge(favorite_category, on="user_id", how="left")
    .merge(customer_events, on="user_id", how="left")
)

customer_table = customer_table.sort_values("user_id").reset_index(drop=True)
customer_table.head()


In [ ]:
customer_table.shape, customer_table.isna().mean().sort_values(ascending=False).head(20)


## Что дальше

Следующий шаг после этого ноутбука:

- определить, кого считаем ушедшим клиентом;
- сформировать целевой признак ухода;
- подготовить обучающую выборку;
- построить первую модель.